In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import pandas as pd
import requests
import wandb

In [ ]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [ ]:
os.makedirs("outputs/prediction", exist_ok=True)

# Table prep functions

In [ ]:
def format_mean_sem(mean, sem, decimals=3, scale=1.0, math_mode=True):
    """Format a single mean/SEM pair as a LaTeX string, with optional scaling."""
    if pd.isna(mean):
        return "--"
    m = mean * scale
    if pd.isna(sem):
        s = f"{m:.{decimals}f}"
    else:
        s = f"{m:.{decimals}f} $\\pm$ {sem * scale:.{decimals}f}"
    return f"{s}" if math_mode else s


def add_mean_sem_columns(df, metrics, math_mode=True):
    """
    metrics: dict mapping output_col_name -> dict with:
        mean_col, sem_col, decimals (int), scale (float, default 1.0)
    """
    df = df.copy()
    for out_col, spec in metrics.items():
        df[out_col] = [
            format_mean_sem(
                m,
                s,
                decimals=spec.get("decimals", 3),
                scale=spec.get("scale", 1.0),
                math_mode=math_mode,
            )
            for m, s in zip(df[spec["mean_col"]], df[spec["sem_col"]])
        ]
    return df


def get_latex(df, columns=None, escape=False):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)

    latex_table = df.to_latex(
        index=False,
        columns=columns,
        escape=escape,  # False: let \pm and $ pass through untouched
    )
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }
    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)
    return latex_table


def parse_modality(x, modalities):
    for k, v in modalities.items():
        if k in x:
            return v

# WANDB API parsing

In [ ]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"

# S2BMS

In [ ]:
project = "s2bms_prediction"
results_df_path = "outputs/prediction/results_s2bms.csv"

## csv log

In [ ]:
if os.path.exists(results_df_path):
    df = pd.read_csv(results_df_path)
    print(f"There are {len(df)} records already in results.")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
else:
    df = None

## fetching

In [ ]:
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue
    elif run.state != "finished":
        print(f"{run.id} is not finished.")
        continue

    captures = dict(run.summary)
    captures["run_id"] = run.id
    captures["seed"] = run.config["seed"]
    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
runs_df = pd.DataFrame(run_list)
if df is not None:
    runs_df = pd.concat([df, runs_df], ignore_index=True)
runs_df.to_csv("outputs/prediction/results_s2bms.csv", index=False)

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    # "best_val_loss": "Validation loss",
    "test_mse_loss": "Test MSE loss",
    # "test_top_1_acc": "Test Top-1 acc",
    "test_top_5_acc": "Test Top-5 acc",
    # "train_mse_loss": "Train MSE loss",
    "test_top_10_acc": "Test Top-10 acc",
    # "best_val_mse_loss": "Validation MSE loss",
    # "best_val_top_1_acc": "Validation Top-1 acc",
    # "best_val_top_5_acc": "Validation Top-5 acc",
    # "best_val_top_10_acc": "Validation Top-10 acc",
}

modalities = {
    "aef": "AlphaEarth",
    "tessera": "Tessera",
    "s2": "Sentinel-2 (RGB)",
    "baseline_mlp": "MLP",
    "baseline_lin": "Linear regression",
    "geoclip": "GeoCLIP",
    "satclip": "SatCLIP",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]
for col in cols_of_interest.keys():
    if "top" in col:
        sub_runs_df[col] = sub_runs_df[col].apply(lambda x: x * 100)

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_mse_loss")
summary.reset_index(inplace=True)

In [ ]:
summary["modality"] = summary["experiment"].apply(lambda x: parse_modality(x, modalities))
cols_of_interest["modality"] = "Modality (best config.)"

In [ ]:
summary

# Table 1

In [ ]:
save_table = True


metrics_s2bms = {
    "Top-10 [\\%]": {
        "mean_col": "test_top_10_acc",
        "sem_col": "test_top_10_acc_sem",
        "decimals": 1,
        "selection_mode": "max",
    },
    "Top-5 [\\%]": {
        "mean_col": "test_top_5_acc",
        "sem_col": "test_top_5_acc_sem",
        "decimals": 1,
        "selection_mode": "max",
    },
    "MSE [$1e{-2}$]": {
        "mean_col": "test_mse_loss",
        "sem_col": "test_mse_loss_sem",
        "decimals": 2,
        "scale": 100.0,
        "selection_mode": "min",
    },
}


def group_table(table, metrics, selection_col="test_mse_loss"):
    selection_mode = [
        metrics[metric]["selection_mode"]
        for metric in metrics
        if metrics[metric]["mean_col"] == selection_col
    ][0]
    if selection_mode == "max":
        tab_1 = table.loc[table.groupby("modality")[selection_col].idxmax()]
    else:
        tab_1 = table.loc[table.groupby("modality")[selection_col].idxmin()]

    return tab_1


tab_1_fmt = group_table(summary, metrics_s2bms, selection_col="test_mse_loss")
tab_1_fmt = tab_1_fmt.rename(columns={"modality": "Modality (best config.)"})
# map display column -> (mean_col, sem_col) present in `summary`


def format_values_table(table, metrics):

    tab_1_fmt = add_mean_sem_columns(table, metrics)

    for output_name, metric_dict in metrics.items():
        metric_val_name = metric_dict["mean_col"]
        ind_best_row = (
            tab_1_fmt[metric_val_name].idxmax()
            if metric_dict["selection_mode"] == "max"
            else tab_1_fmt[metric_val_name].idxmin()
        )
        tab_1_fmt.loc[ind_best_row, output_name] = (
            "\\textbf{" + tab_1_fmt.loc[ind_best_row, output_name] + "}"
        )

    return tab_1_fmt


tab_1_fmt = format_values_table(tab_1_fmt, metrics_s2bms)

tab_1_print = tab_1_fmt[["Modality (best config.)"] + list(metrics_s2bms.keys())]


## Insert row:
tab_1_print.loc[len(tab_1_print)] = ["Mean rate*", "67.3", "58.7", "1.39"]
tab_1_print.loc[len(tab_1_print)] = [
    "Sentinel-2 (RGB + IR) + contrastive [SOTA]*",
    "70.4 $\\pm$ 0.2",
    "63.0 $\\pm$ 0.9",
    "1.20 $\\pm$ 0.03",
]

order_modalities = [
    "Mean rate*",
    "Sentinel-2 (RGB)",
    "Sentinel-2 (RGB + IR) + contrastive [SOTA]*",
    "Linear regression",
    "MLP",
    "GeoCLIP",
    "SatCLIP",
    "Tessera",
    "AlphaEarth",
]
tab_1_print = (
    tab_1_print.set_index("Modality (best config.)").reindex(order_modalities).reset_index()
)
if save_table:
    tab_1_print.to_latex(
        "/Users/tplas/repos/ms_aether_biodiv/tables/pred_s2bms_summary.tex",
        index=False,
        escape=False,
        label="tab:pred_s2bms_summary",
        caption="\\textbf{AlphaEarth is the best modality for the S2BMS task.} The best modality for each metric is highlighted in bold. The metrics are top-5 and top-10 accuracy (higher is better) and mean squared error (MSE) (lower is better). Results from \\cite{vanderplas2025predicting} are marked with *.",
    )
    # \\citep{vanderplas2025predicting}

tab_1_print

# Table S1 per modality

In [ ]:
pred_per_modality = {}
for m in modalities.values():
    tab_s1 = summary[summary["modality"] == m]
    assert (
        tab_s1["n_runs"].nunique() == 1 and tab_s1["n_runs"].iloc[0] == 3
    ), f"Modality {m} has different number of runs for different experiments."
    # tab_s1.drop(columns=['n_runs'])
    pred_per_modality[m] = tab_s1

In [ ]:
save_table = True

for m in ["AlphaEarth", "Tessera"]:
    ## spatial pooling (av / rn / n/a)
    ## size (1 / 128 / 256)
    ## hidden layers (0, 1, 2)

    pred_tab = pred_per_modality[m]
    pred_tab["Pooling"] = pred_tab["experiment"].apply(
        lambda x: (
            "n/a"
            if "_1_" in x
            else ("resnet-" + x.split("rn")[1].split("_")[0] if "rn" in x else "average")
        )
    )
    pred_tab["Data size"] = pred_tab["experiment"].apply(
        lambda x: (
            x.split("tessera_")[1].split("_")[0]
            if "tessera" in x
            else (x.split("aef_")[1].split("_")[0] if "aef" in x else "n/a")
        )
    )
    pred_tab["Hidden layers"] = pred_tab["experiment"].apply(
        lambda x: "0" if "linear" in x else ("2" if "_deeper_" in x else "1")
    )

    pred_tab = (
        format_values_table(pred_tab, metrics_s2bms)[
            ["Pooling", "Data size", "Hidden layers"] + list(metrics_s2bms.keys())
        ]
        .sort_values(["Data size", "Pooling", "Hidden layers"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_s2bms_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_s2bms_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for {m} on the S2BMS task.}} The best configuration for {m} is highlighted in bold. The metrics are top-5 and top-10 accuracy (higher is better) and mean squared error (MSE) (lower is better).",
        )

In [ ]:
save_table = True

for m in ["MLP"]:
    ## hidden size (128 / 256)
    ## hidden layers (2, 3)

    pred_tab = pred_per_modality[m]
    pred_tab["Hidden units"] = pred_tab["experiment"].apply(
        lambda x: "128" if "mlp128_" in x else ("256" if "mlp256_" in x else "n/a")
    )
    pred_tab["Hidden layers"] = pred_tab["experiment"].apply(lambda x: "2" if "_2" in x else "3")

    pred_tab = (
        format_values_table(pred_tab, metrics_s2bms)[
            ["Hidden units", "Hidden layers"] + list(metrics_s2bms.keys())
        ]
        .sort_values(["Hidden units", "Hidden layers"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_s2bms_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_s2bms_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for {m} on the S2BMS task.}} The best configuration for {m} is highlighted in bold. The metrics are top-5 and top-10 accuracy (higher is better) and mean squared error (MSE) (lower is better).",
        )

In [ ]:
save_table = True

for m in ["Sentinel-2 (RGB)"]:
    ## data size (128 / 256)
    ## channels (rgb / 4c)
    ## encoder (rn)
    ## pretrained (ssl40eo ,  .. )

    pred_tab = pred_per_modality[m]
    pred_tab["Data size"] = pred_tab["experiment"].apply(lambda x: x.split("_")[1])
    pred_tab["Channels"] = pred_tab["experiment"].apply(
        lambda x: "RGB" if "rgb" in x else ("RGB + IR" if "4c" in x else "n/a")
    )
    pred_tab["Encoder"] = pred_tab["experiment"].apply(
        lambda x: "resnet-" + x.split("rn")[1].split("_")[0] if "rn" in x else "n/a"
    )
    pred_tab["Pretrained"] = pred_tab["experiment"].apply(
        lambda x: (
            "Frozen SSL4EO"
            if "frozen_ssl4eo" in x
            else ("SSL4EO" if "ssl4eo" in x else ("ImageNet" if "imagenet" in x else "No"))
        )
    )

    pred_tab = (
        format_values_table(pred_tab, metrics_s2bms)[
            ["Data size", "Channels", "Encoder", "Pretrained"] + list(metrics_s2bms.keys())
        ]
        .sort_values(["Data size", "Channels", "Encoder", "Pretrained"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_s2bms_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_s2bms_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for Sentinel-2 on the S2BMS task.}} The best configuration for {m} is highlighted in bold. The metrics are top-5 and top-10 accuracy (higher is better) and mean squared error (MSE) (lower is better).",
        )

In [ ]:
save_table = True

for m in ["GeoCLIP", "SatCLIP"]:
    ## mlp

    pred_tab = pred_per_modality[m]
    pred_tab["Hidden layers"] = pred_tab["experiment"].apply(
        lambda x: "2" if "deeper" in x else ("1" if "mlp" in x else "0")
    )
    pred_tab = (
        format_values_table(pred_tab, metrics_s2bms)[
            ["Hidden layers"] + list(metrics_s2bms.keys())
        ]
        .sort_values(["Hidden layers"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_s2bms_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_s2bms_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for {m} on the S2BMS task.}} The best configuration for {m} is highlighted in bold. The metrics are top-5 and top-10 accuracy (higher is better) and mean squared error (MSE) (lower is better).",
        )


pred_tab

# Satbird

In [ ]:
project = "satbird-usa-summer_prediction"
results_df_path = "outputs/prediction/results_satbird.csv"

## csv log

In [ ]:
if os.path.exists(results_df_path):
    df = pd.read_csv(results_df_path)
    print(f"There are {len(df)} records already in results.")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
else:
    df = None

## fetching

In [ ]:
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue
    elif run.state != "finished":
        print(f"{run.id} is not finished.")
        continue

    captures = dict(run.summary)
    captures["run_id"] = run.id
    captures["seed"] = run.config["seed"]
    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
runs_df = pd.DataFrame(run_list)
if df is not None:
    runs_df = pd.concat([df, runs_df], ignore_index=True)
runs_df.to_csv("outputs/prediction/results_satbird.csv", index=False)

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "test_mse_loss": "Test MSE loss",
    "test_mae_loss": "Test MAE loss",
    "test_top_30_acc": "Test Top-30 acc",
    "test_top_10_acc": "Test Top-10 acc",
}

modalities = {
    "aef": "AlphaEarth",
    "baseline_mlp": "MLP",
    "baseline_lin": "Linear regression",
    "geoclip": "GeoCLIP",
    "satclip": "SatCLIP",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]
for col in cols_of_interest.keys():
    if "top" in col:
        sub_runs_df[col] = sub_runs_df[col].apply(lambda x: x * 100)

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_mse_loss")
summary.reset_index(inplace=True)

In [ ]:
summary["modality"] = summary["experiment"].apply(lambda x: parse_modality(x, modalities))
cols_of_interest["modality"] = "Modality (best config.)"

In [ ]:
summary

# Table 1

In [ ]:
metrics_satbird = {
    "Top-10 [\\%]": {
        "mean_col": "test_top_10_acc",
        "sem_col": "test_top_10_acc_sem",
        "decimals": 1,
        "selection_mode": "max",
    },
    "Top-30 [\\%]": {
        "mean_col": "test_top_30_acc",
        "sem_col": "test_top_30_acc_sem",
        "decimals": 1,
        "selection_mode": "max",
    },
    "MSE [$1e{-2}$]": {
        "mean_col": "test_mse_loss",
        "sem_col": "test_mse_loss_sem",
        "decimals": 2,
        "scale": 100.0,
        "selection_mode": "min",
    },
    "MAE [$1e{-2}$]": {
        "mean_col": "test_mae_loss",
        "sem_col": "test_mae_loss_sem",
        "decimals": 2,
        "scale": 100.0,
        "selection_mode": "min",
    },
}

selection_col = "test_mse_loss"
save_table = True

tab_1_fmt = group_table(summary, metrics_satbird, selection_col="test_mse_loss")
tab_1_fmt = tab_1_fmt.rename(columns={"modality": "Modality (best config.)"})
tab_1_fmt = format_values_table(tab_1_fmt, metrics_satbird)

tab_1_print = tab_1_fmt[["Modality (best config.)"] + list(metrics_satbird.keys())]
tab_1_print = tab_1_print.reset_index(drop=True)
# # ## Insert row 10, 30, mse , mae
tab_1_print.loc[len(tab_1_print)] = [
    "Sentinel-2 RGB + env. [SOTA]*",
    "46.3 $\\pm$ 0.2",
    "\\textbf{65.5 $\\pm$ 0.2}",
    "0.64 $\\pm$ 0.0",
    "2.2 $\\pm$ 0.02",
]
tab_1_print.loc[len(tab_1_print)] = [
    "Sentinel-2 RGB*",
    "35.5 $\\pm$ 0.1",
    "52.2 $\\pm$ 0.2",
    "0.8 $\\pm$ 0.0",
    "2.6 $\\pm$ 0.02",
]
tab_1_print.loc[len(tab_1_print)] = ["Mean rate*", "26.9", "38.6", "0.9", "3.1"]

# ## replace AlphaEarth top-30 value with non textbf value
tab_1_print.loc[tab_1_print["Modality (best config.)"] == "AlphaEarth", "Top-30 [\\%]"] = (
    tab_1_print.loc[
        tab_1_print["Modality (best config.)"] == "AlphaEarth", "Top-30 [\\%]"
    ].str.replace(r"\\textbf\{(.*)\}", r"\1", regex=True)
)


order_modalities = [
    "Mean rate*",
    "Sentinel-2 RGB*",
    "Sentinel-2 RGB + env. [SOTA]*",
    "Linear regression",
    "MLP",
    "GeoCLIP",
    "SatCLIP",
    "AlphaEarth",
]

tab_1_print = (
    tab_1_print.set_index("Modality (best config.)").reindex(order_modalities).reset_index()
)

if save_table:
    tab_1_print.to_latex(
        "/Users/tplas/repos/ms_aether_biodiv/tables/pred_satbird_summary.tex",
        index=False,
        escape=False,
        label="tab:pred_satbird_summary",
        caption="\\textbf{AlphaEarth is the best modality for the Satbird-USA-Summer task.} The best modality for each metric is highlighted in bold. The metrics are top-10 and top-30 accuracy (higher is better) and mean squared/absolute error (MSE/MAE) (lower is better). Results from \\cite{teng2023satbird} are marked with *.",
    )

tab_1_print

# Table S1 per modality

In [ ]:
pred_per_modality = {}
for m in modalities.values():
    tab_s1 = summary[summary["modality"] == m]
    assert (
        tab_s1["n_runs"].nunique() == 1 and tab_s1["n_runs"].iloc[0] == 3
    ), f"Modality {m} has different number of runs for different experiments."
    # tab_s1.drop(columns=['n_runs'])
    pred_per_modality[m] = tab_s1

In [ ]:
pred_per_modality["AlphaEarth"]

In [ ]:
save_table = True

for m in ["AlphaEarth"]:
    ## spatial pooling (av )
    ## size (128 / 256)
    ## hidden layers (0, 1)

    pred_tab = pred_per_modality[m]
    pred_tab["Pooling"] = pred_tab["experiment"].apply(
        lambda x: (
            "n/a"
            if "_1_" in x
            else ("resnet-" + x.split("rn")[1].split("_")[0] if "rn" in x else "average")
        )
    )
    pred_tab["Data size"] = pred_tab["experiment"].apply(
        lambda x: (
            x.split("tessera_")[1].split("_")[0]
            if "tessera" in x
            else (x.split("aef_")[1].split("_")[0] if "aef" in x else "n/a")
        )
    )
    pred_tab["Hidden layers"] = pred_tab["experiment"].apply(
        lambda x: "0" if "linear" in x else ("2" if "_deeper_" in x else "1")
    )

    pred_tab = (
        format_values_table(pred_tab, metrics_satbird)[
            ["Pooling", "Data size", "Hidden layers"] + list(metrics_satbird.keys())
        ]
        .sort_values(["Data size", "Pooling", "Hidden layers"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_satbird_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_satbird_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for {m} on Satbird-USA-Summer.}} The best configuration for {m} is highlighted in bold. The metrics are top-10 and top-30 accuracy (higher is better) and mean squared/absolute error (MSE/MAE) (lower is better).",
        )


pred_tab

In [ ]:
save_table = True

for m in ["MLP"]:
    ## hidden size (128 / 256)
    ## hidden layers (2, 3)

    pred_tab = pred_per_modality[m]
    pred_tab["Hidden units"] = pred_tab["experiment"].apply(
        lambda x: "128" if "mlp128_" in x else ("256" if "mlp256_" in x else "n/a")
    )
    pred_tab["Hidden layers"] = pred_tab["experiment"].apply(lambda x: "2" if "_2" in x else "3")

    pred_tab = (
        format_values_table(pred_tab, metrics_satbird)[
            ["Hidden units", "Hidden layers"] + list(metrics_satbird.keys())
        ]
        .sort_values(["Hidden units", "Hidden layers"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_satbird_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_satbird_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for {m} on Satbird-USA-Summer.}} The best configuration for {m} is highlighted in bold. The metrics are top-10 and top-30 accuracy (higher is better) and mean squared/absolute error (MSE/MAE) (lower is better).",
        )

pred_tab

In [ ]:
save_table = True

for m in ["GeoCLIP", "SatCLIP"]:
    ## mlp

    pred_tab = pred_per_modality[m]
    pred_tab["Hidden layers"] = pred_tab["experiment"].apply(
        lambda x: "2" if "deeper" in x else ("1" if "mlp" in x else "0")
    )
    pred_tab = (
        format_values_table(pred_tab, metrics_satbird)[
            ["Hidden layers"] + list(metrics_satbird.keys())
        ]
        .sort_values(["Hidden layers"])
        .reset_index(drop=True)
    )

    if save_table:
        pred_tab.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/pred_satbird_{m.lower()}_ablations.tex",
            index=False,
            escape=False,
            label=f"tab:pred_satbird_{m.lower()}_ablations",
            caption=f"\\textbf{{Ablation study for {m} on Satbird-USA-Summer.}} The best configuration for {m} is highlighted in bold. The metrics are top-10 and top-30 accuracy (higher is better) and mean squared/absolute error (MSE/MAE) (lower is better).",
        )


pred_tab